In [123]:
# Program to find origin, provider, prefix, its length and version from raw data generated using bgpreader as 
# bgpreader -t ribs -w 1704067200,1704067200 -A _19551_ >> /home/shyam/jupy/ddos_scrubber/data/raw_as19551_01_jan_2024.txt
# Use python multicore programming feature

import pandas as pd
import csv
from concurrent.futures import ProcessPoolExecutor

def process_chunk(lines):
    """Process a chunk of lines and extract prefix, AS path, and origin AS."""
    chunk_data = []
    for line in lines:
        fields = line.strip().split('|')
        if len(fields) > 12 and fields[1] == "R":
            try:
               # Extract prefix, AS path, and origin AS
                prefix = fields[9]
                as_path = fields[11].split()


                # Find provider here checking AS repetetitions and the same organization owning multiple ASes
#                 provider = find_immediate_provider(as_path)
                
                provider = as_path[-2] if len(as_path) > 1 else None  # The ASN before the origin AS
                origin_as = as_path[-1] if as_path else None
                
                pfx_len = int(prefix.split('/')[1]) if '/' in prefix else None
                ip_version = "IPv6" if ':' in prefix else "IPv4"

                # Append data to the chunk
                chunk_data.append([prefix, ' '.join(as_path), origin_as, provider, pfx_len, ip_version])
            
            except IndexError:
                continue
    return chunk_data

def process_route_data_parallel(file_path, output_file, num_workers=8, chunk_size=100000):
    with open(file_path, 'r') as file:
        lines = []
        
        # Initialize CSV output
        with open(output_file, 'w', newline='') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['prefix', 'as_path', 'origin_as', 'provider', 'pfx_len', 'ip_version'])

        # Process the file in chunks with multiple workers
        with ProcessPoolExecutor(max_workers=num_workers) as executor:
            futures = []
            for line in file:
                lines.append(line)
                
                # When lines reach chunk size, process them
                if len(lines) >= chunk_size:
                    futures.append(executor.submit(process_chunk, lines))
                    lines = []

            # Process any remaining lines after the loop
            if lines:
                futures.append(executor.submit(process_chunk, lines))

            # Collect results from all futures and write to CSV
            for future in futures:
                chunk_data = future.result()
                if chunk_data:
                    df = pd.DataFrame(chunk_data, columns=['prefix', 'as_path', 'origin_as', 'provider', 'pfx_len', 'ip_version'])
                    df.to_csv(output_file, mode='a', header=False, index=False)

    print(f"Data successfully saved to {output_file}")

# Example usage
file_path = '/home/shyam/jupy/ddos_scrubber/data/raw_as19551_01_jan_2017.txt'
output_file = '/home/shyam/jupy/ddos_scrubber/data/optimized_raw_as19551_01_jan_2017.csv'
process_route_data_parallel(file_path, output_file)


Data successfully saved to /home/shyam/jupy/ddos_scrubber/data/optimized_raw_as19551_01_jan_2017.csv


In [124]:
# Read csv file and remove duplicate rows
import pandas as pd
df = pd.read_csv('/home/shyam/jupy/ddos_scrubber/data/optimized_raw_as19551_01_jan_2017.csv', low_memory=False)
 
# Remove duplicate rows in dataframe
df = df.drop_duplicates()

# Write unique rows into a file
df.to_csv('/home/shyam/jupy/ddos_scrubber/data/unique_optimized_raw_as19551_01_jan_2017.csv', index=False)
df

,prefix,as_path,origin_as,provider,pfx_len,ip_version
0,27.50.0.0/24,3257 4436 19551 19551 19551 19551 19551 19551 ...,55303,19551,24,IPv4
1,27.50.3.0/24,3292 3257 4436 19551 19551 19551 19551 19551 1...,55303,19551,24,IPv4
2,27.50.3.0/24,286 174 19551 19551 19551 19551 19551 19551 19...,55303,19551,24,IPv4
3,27.50.3.0/24,10026 174 19551 19551 19551 19551 19551 19551 ...,55303,19551,24,IPv4
4,27.50.3.0/24,40387 23473 6939 19551 19551 19551 19551 55303,55303,19551,24,IPv4
...,...,...,...,...,...,...
47682,27.50.3.0/24,22548 16735 174 19551 19551 19551 19551 19551 ...,55303,19551,24,IPv4
47683,27.50.3.0/24,25933 3549 3356 174 19551 19551 19551 19551 19...,55303,19551,24,IPv4
47685,27.50.3.0/24,28220 6762 174 19551 19551 19551 19551 19551 1...,55303,19551,24,IPv4
47687,27.50.3.0/24,53070 10429 12956 174 19551 19551 19551 19551 ...,55303,19551,24,IPv4


In [125]:
# Remove records containing AS set on an AS path
import pandas as pd
df = pd.read_csv('/home/shyam/jupy/ddos_scrubber/data/unique_optimized_raw_as19551_01_jan_2017.csv', low_memory=False)
print("Records before %s." %len(df)) 

print("Removing AS set in an AS path")
# Find rows containing '{}' (set origins)
mask = df['as_path'].str.contains(r'\{.*\}')
rows_with_origin_set = df[mask]

# Count rows with set origin
count_set_origin = rows_with_origin_set.shape[0]

# Remove rows with set origin from the DataFrame
df_cleaned = df[~mask]

# Remove rows with set origin from the DataFrame
df_cleaned = df[~mask]

df_cleaned.to_csv('/home/shyam/jupy/ddos_scrubber/data/unique_optimized_raw_as19551_01_jan_2017.csv', index = False)
print("Number of rows with set origin:", count_set_origin)
print("DataFrame after removal:")
print("Records after %s." %len(df_cleaned)) 
print("%s number of records were removed that contains AS set in AS paths." %(len(df) - len(df_cleaned)))


Records before 41687.
Removing AS set in an AS path
Number of rows with set origin: 0
DataFrame after removal:
Records after 41687.
0 number of records were removed that contains AS set in AS paths.


In [126]:
import pandas as pd
df = pd.read_csv('/home/shyam/jupy/ddos_scrubber/data/unique_optimized_raw_as19551_01_jan_2017.csv', low_memory=False)
df

,prefix,as_path,origin_as,provider,pfx_len,ip_version
0,27.50.0.0/24,3257 4436 19551 19551 19551 19551 19551 19551 ...,55303,19551,24,IPv4
1,27.50.3.0/24,3292 3257 4436 19551 19551 19551 19551 19551 1...,55303,19551,24,IPv4
2,27.50.3.0/24,286 174 19551 19551 19551 19551 19551 19551 19...,55303,19551,24,IPv4
3,27.50.3.0/24,10026 174 19551 19551 19551 19551 19551 19551 ...,55303,19551,24,IPv4
4,27.50.3.0/24,40387 23473 6939 19551 19551 19551 19551 55303,55303,19551,24,IPv4
...,...,...,...,...,...,...
41682,27.50.3.0/24,22548 16735 174 19551 19551 19551 19551 19551 ...,55303,19551,24,IPv4
41683,27.50.3.0/24,25933 3549 3356 174 19551 19551 19551 19551 19...,55303,19551,24,IPv4
41684,27.50.3.0/24,28220 6762 174 19551 19551 19551 19551 19551 1...,55303,19551,24,IPv4
41685,27.50.3.0/24,53070 10429 12956 174 19551 19551 19551 19551 ...,55303,19551,24,IPv4


In [127]:
import re
from collections import defaultdict

def count_prefix_lengths_and_versions(prefixes):
    # Patterns to match IPv4 and IPv6
    ipv4_pattern = re.compile(r"^(?:\d{1,3}\.){3}\d{1,3}$")
    ipv6_pattern = re.compile(r"^(?:[a-fA-F0-9]{1,4}:){1,7}[a-fA-F0-9]{1,4}$|^(?:[a-fA-F0-9]{0,4}:){1,7}:$|^::(?:[a-fA-F0-9]{1,4}:){0,6}[a-fA-F0-9]{1,4}$")
    
    # Dictionary to hold counts based on IP version and prefix length
    counts = defaultdict(lambda: defaultdict(int))

    for prefix in prefixes:
        # Split into IP address and prefix length
        ip, length = prefix.split('/')
        
        # Detect IP version and update counts
        if ipv4_pattern.match(ip):
            ip_version = "IPv4"
        elif ipv6_pattern.match(ip):
            ip_version = "IPv6"
        else:
            continue  # Ignore invalid IPs if any

        # Update counts
        counts[ip_version][length] += 1

    return counts

In [128]:
# Find records that have origin differnet from AS19551 and their siblings; and contain AS19551 as 
# the second last hop AS on their AS paths

print("%s number of records in RIB file that have AS19551 on their AS Paths" %len(df))
# Find number of unique prefixes that contain AS19551 as origin
condition = (df['origin_as'] == 19551) 
df2 = df.loc[condition]
print("%s number of prefixes have origin AS19551" %len(df2))

# Find number of unique prefixes that do not contain AS19551 as well as its siblings as origin
# Siblings are found using ASRank API for the date 2024-01-01
siblings = [19551]

condition = (~df['origin_as'].isin (siblings)) 
df2 = df.loc[condition]
print("%s number of prefixes do not have origin AS19551 and its siblings" %len(df2))

# Prefixes containing scrubber asn as the second last hop AS in AS path
provider_scrubber_records = df2.loc[df['provider'] == 19551]
print("%s number of prefixes contain AS19551 as a provider(second last hop in AS path)" %len(provider_scrubber_records))

# Save confirmed_costumers1 to a csv file
provider_scrubber_records.to_csv("/home/shyam/jupy/ddos_scrubber/data/confirmed_customers_as19551_2017.csv", index=False)
# Unique origin ASes are actually CONFIRMED customers that have their second last hop AS as AS19551
unique_origin_ases = provider_scrubber_records['origin_as'].unique()
print("%s number of unique customer ASNs of AS19551" %len(unique_origin_ases))

unique_prefixes = provider_scrubber_records['prefix'].unique()
print("%s number of unique prefixes that contain AS19551 as provider" %len(unique_prefixes))

# # Find prefixes sizes, their numbers and IP versions
result = count_prefix_lengths_and_versions(unique_prefixes)

# Print results
for version, lengths in result.items():
    print(f"{version}:")
    for length, count in lengths.items():
        print(f"  /{length}: {count} prefixes")

41687 number of records in RIB file that have AS19551 on their AS Paths
27591 number of prefixes have origin AS19551
14096 number of prefixes do not have origin AS19551 and its siblings
11771 number of prefixes contain AS19551 as a provider(second last hop in AS path)
40 number of unique customer ASNs of AS19551
240 number of unique prefixes that contain AS19551 as provider
IPv4:
  /24: 230 prefixes
  /23: 10 prefixes


In [129]:
# Look for the cases where records contain the second last hop AS other than AS19551 => Semi-confirmed customers
# Probable reasons: 1. AS path prepending 2. Sibling ASes
# 1. AS path prepending
import pandas as pd
# df = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_as19551_01_jan_2024.csv", low_memory=False)
provider_not_scrubber = df2.loc[df['provider'] != 19551]
provider_not_scrubber.to_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_not_as19551_01_jan_2017.csv", index=False)
print("%s number of prefixes do not contain AS19551 as a provider (second last hop in AS path)." %len(provider_not_scrubber))

2325 number of prefixes do not contain AS19551 as a provider (second last hop in AS path).


In [130]:
# CASE #1: AS path prepending for the records containing the second last hop AS other than AS19551 => Semi-confirmed customers 
import pandas as pd

# Define a function to determine path prepending and new_provider
def determine_path_prepending_and_new_provider(row):
   
    # Ensure as_path is a string and split into a list of ASNs
    as_path = list(map(int, str(row['as_path']).split()))
    provider = None # Default value set
    if len(as_path) < 2:
        provider = None  # Not enough ASNs in path to determine provider

    origin_as = as_path[-1]  # The last ASN is the origin ASN
    
    path_prepending = 1 if as_path.count(origin_as) > 1 else 0

#     provider = as_path[-2]  # If all checks fail, return the second ASN by default
    
    # Check for sequentially repeated ASNs
    repeated_asn = None
    for i in range(len(as_path) - 1, 0, -1):
        if as_path[i-1] == as_path[i]:
            repeated_asn = as_path[i]
        else:
            # If we find an ASN that is not the same as the repeated ASN,
            # and we have found a repeated ASN, return the one before the repeated ASN
            if repeated_asn is not None:
                provider = as_path[i-1]  # This is the upstream provider
            break
    
    return pd.Series([path_prepending, provider])

df = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_not_as19551_01_jan_2017.csv")
# Apply the function to each row
df[['path_prepending', 'new_provider']] = df.apply(determine_path_prepending_and_new_provider, axis=1)

# Print the updated DataFrame to verify results
df.to_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_not_as19551_01_jan_2017_v2.csv") 

# Find the records of AS path prepending
condition = (df['path_prepending'] == 1) & (df['new_provider'] == 19551)
provider_scrubber_as_path_prepend = df.loc[condition]
print("%s number of records have AS path prepended and contain AS19551 as a provider(not a second last hop in AS path though)" %len(provider_scrubber_as_path_prepend))
provider_scrubber_as_path_prepend.to_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_as19551_path_prepend_01_jan_2017.csv")

1813 number of records have AS path prepended and contain AS19551 as a provider(not a second last hop in AS path though)


In [131]:
test = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_not_as19551_01_jan_2017_v2.csv")
test.loc[test["path_prepending"] == 1] 

,Unnamed: 0,prefix,as_path,origin_as,provider,pfx_len,ip_version,path_prepending,new_provider
0,0,27.125.204.0/22,3561 209 3491 19551 19551 19551 19551 55383 55383,55383,55383,22,IPv4,1.0,19551.0
1,1,27.125.204.0/22,286 3491 19551 19551 19551 19551 55383 55383,55383,55383,22,IPv4,1.0,19551.0
2,2,27.125.204.0/22,3292 3491 19551 19551 19551 19551 55383 55383,55383,55383,22,IPv4,1.0,19551.0
3,3,27.125.204.0/22,10026 3491 19551 19551 19551 19551 55383 55383,55383,55383,22,IPv4,1.0,19551.0
4,4,27.125.204.0/22,40387 2381 3491 19551 19551 19551 19551 55383 ...,55383,55383,22,IPv4,1.0,19551.0
...,...,...,...,...,...,...,...,...,...
2320,2320,27.125.205.0/24,25152 1273 2914 19551 19551 19551 19551 55383 ...,55383,55383,24,IPv4,1.0,19551.0
2321,2321,27.125.206.0/24,20080 2914 19551 19551 19551 19551 55383 55383,55383,55383,24,IPv4,1.0,19551.0
2322,2322,27.125.206.0/24,25152 1273 2914 19551 19551 19551 19551 55383 ...,55383,55383,24,IPv4,1.0,19551.0
2323,2323,27.125.207.0/24,20080 2914 19551 19551 19551 19551 55383 55383,55383,55383,24,IPv4,1.0,19551.0


In [46]:
# CASE #2: Check if there exists sibling ASes of the origin AS => Semi-confirmed customers 2 
# Do it in Aruba machine
# ssh -L 8900:localhost:8900 aruba-shyam 
 

# Then in your local browser go to: http://127.0.0.1:8900/lab/workspaces/ 

# Save your code under workspace/notebooks 

# http://127.0.0.1:8900/lab?token=9a3db0bf114d31779f0df8dc005838c143a39e5e70c8ab73         
        
import pandas as pd
from concurrent.futures import ProcessPoolExecutor

# Function to determine sibling ASes and new provider
def determine_sibling_ases_and_new_provider(row):
    # Ensure as_path is a string and split into a list of ASNs
    as_path = list(map(int, str(row['as_path']).split()))
    
    # If as_path is too short to determine provider, return default values
    if len(as_path) < 2:
        return (0, None)  # No siblings, no provider

    origin_as = as_path[-1]  # The last ASN is the origin ASN
    provider = as_path[-2]  # Default provider is the second-to-last ASN
    
    # Default sibling count
    siblings = 0 
    
    # Retrieve sibling list from an API (or mock it here for testing)
    try:
        sibling_list = find_siblings(str(origin_as))
    except Exception as e:
        print(f"Error retrieving siblings: {e}")
        return (0, None)  # No siblings, no provider in case of API failure

    # Traverse the AS path to identify a non-sibling provider
    for i in range(len(as_path) - 1, 0, -1):
        if str(as_path[i-1]) not in sibling_list:
            provider = as_path[i-1]
            break

    # Determine if there are any siblings in the AS path
    as_path_str = list(map(str, as_path))  # Convert AS path to strings for comparison
    siblings = 1 if len(set(as_path_str) & set(sibling_list)) > 0 else 0

    return (siblings, provider)  # Return as tuple

# Function to process a chunk of the DataFrame
def process_chunk(chunk):
    # Apply the function and collect results
    results = chunk.apply(determine_sibling_ases_and_new_provider, axis=1)
    
    # Convert results to DataFrame with correct columns
    results_df = pd.DataFrame(results.tolist(), columns=['siblings', 'new_provider_sibling_check'], index=chunk.index)
    
    return results_df

# Load dataset and filter by specific criteria
df = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_not_as19551_01_jan_2024_v2.csv")
# Find the cases where no path prepending exists and prefixes are not originated by the scrubber 
df = df.loc[df['path_prepending'] == 0]

print(f"{len(df)} records found with AS path prepending = 0 and provider not AS19551.")

# Define chunk size and split data into chunks for parallel processing
chunk_size = 1000  # Adjust based on available CPU cores and memory
chunks = [df[i:i + chunk_size] for i in range(0, len(df), chunk_size)]

# Set the number of workers (processors)
num_workers = 35  # Adjust this number based on your CPU

# Use ProcessPoolExecutor for parallel processing
with ProcessPoolExecutor(max_workers=num_workers) as executor:
    # Map the function over each chunk and gather results
    results = list(executor.map(process_chunk, chunks))

# Combine processed chunks into a single DataFrame with reset index
processed_chunks = pd.concat(results)

# Assign processed data to the original DataFrame columns
df[['siblings', 'new_provider_sibling_check']] = processed_chunks

# Print final output or save to file
df.to_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_not_as19551_01_jan_2024_v3.csv")

6044 records found with AS path prepending = 0 and provider not AS19551.
Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not defined


Error retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined



Error retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings'

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not defined


Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not defined


Error retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not 

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not 

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not defined


Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not defined



Error retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not de

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined


Error retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined


Error retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not de

...

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined


Error retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not defined


Error retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not de

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined


Error retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined


Error retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not d

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined





Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not de

Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not defined

Error retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not defined



Error retrieving siblings: name 'find_siblings' is not defined
Error retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not definedError retrieving siblings: name 'find_siblings' is not def

In [132]:
# Find the number of records containing siblings and not siblings. 

import pandas as pd
df = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_not_as19551_01_jan_2017_v3.csv")

# Confirmed customers (AS19551 comes immediately after the siblings) after sibling check
confirmed_customers_after_sibling_check = df.loc[(df['siblings'] == 1) & (df['new_provider_sibling_check'] == 19551)]
print("%s number of records have sibling ASes and contain AS19551 as a provider (not a second last hop in AS path though)" %len(confirmed_customers_after_sibling_check))

no_sibling_different_provider_records = df.loc[(df['siblings'] == 0) & (df['new_provider_sibling_check'] != 19551)]
print("%s number of records do not have sibling ASes and contain another provider" %len(no_sibling_different_provider_records))
no_sibling_different_provider_records

0 number of records have sibling ASes and contain AS19551 as a provider (not a second last hop in AS path though)
457 number of records do not have sibling ASes and contain another provider


,Unnamed: 0.1,Unnamed: 0,prefix,as_path,origin_as,provider,pfx_len,ip_version,path_prepending,new_provider,siblings,new_provider_sibling_check
0,104,104,103.17.184.0/22,53828 6939 3491 19551 19551 19551 19551 55383 ...,58654,55383,22,IPv4,0.0,NaN,0,55383
1,105,105,103.17.184.0/22,19016 3257 3491 19551 19551 19551 19551 55383 ...,58654,55383,22,IPv4,0.0,NaN,0,55383
2,106,106,103.17.184.0/22,32709 6939 3491 19551 19551 19551 19551 55383 ...,58654,55383,22,IPv4,0.0,NaN,0,55383
3,107,107,103.17.184.0/22,19653 6461 2914 19551 19551 19551 19551 55383 ...,58654,55383,22,IPv4,0.0,NaN,0,55383
4,108,108,103.17.184.0/22,852 3491 19551 19551 19551 19551 55383 58654,58654,55383,22,IPv4,0.0,NaN,0,55383
...,...,...,...,...,...,...,...,...,...,...,...,...
452,2310,2310,185.175.100.0/23,15301 6939 3491 19551 19551 19551 19551 134785...,206819,134785,23,IPv4,0.0,NaN,0,134785
453,2311,2311,185.175.102.0/23,6423 209 3491 19551 19551 19551 19551 134785 2...,206819,134785,23,IPv4,0.0,NaN,0,134785
454,2312,2312,185.175.102.0/23,1798 174 3491 19551 19551 19551 19551 134785 2...,206819,134785,23,IPv4,0.0,NaN,0,134785
455,2313,2313,185.175.102.0/23,54073 2044 174 3491 19551 19551 19551 19551 13...,206819,134785,23,IPv4,0.0,NaN,0,134785


In [133]:
# Find customers of 19551 where it is not an immediate ASN e.g. 1798 6461 19551 43338 44815
# It might be used to validate our method of finding the number of customers with AS rank customer ASNs
import pandas as pd

# Main thing is to find the location of the scrubber in an AS path
# Function to extract ASes after 19551
def find_ases_after_19551(as_path):
    ases = as_path.split()
    if '19551' in ases:
        idx = ases.index('19551')
        return ases[idx + 1:]
    return []

# Apply the function to the as_path column and get a unique list of ASes
customer_asns_not_sibling_not_immediate_provider = no_sibling_different_provider_records['as_path'].apply(find_ases_after_19551).explode().unique().tolist()

print("%s ASes lie after AS19551, not as the second last hop in their AS paths."%len(customer_asns_not_sibling_not_immediate_provider))


8 ASes lie after AS19551, not as the second last hop in their AS paths.


In [134]:
# Compare confirmed_customers_after_sibling_check, path prepending with the confirmed customers
import pandas as pd

# After checking second last hop AS as AS19551
confirmed_customers1 = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/confirmed_customers_as19551_2017.csv")
confirmed_customers1_unique_origin_ases = confirmed_customers1['origin_as'].unique()
confirmed_customers1_unique_prefixes = confirmed_customers1['prefix'].unique()

# After checking path prepending
confirmed_customers2 = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_as19551_path_prepend_01_jan_2017.csv")
confirmed_customers2_unique_origin_ases = confirmed_customers2['origin_as'].unique()
confirmed_customers2_unique_prefixes = confirmed_customers2['prefix'].unique()

ases_sometimes_prepend_not_prepend = set(confirmed_customers1_unique_origin_ases) & set(confirmed_customers2_unique_origin_ases)
print("%s number of ASes sometimes prepend their AS numbers and sometimes not for their different prefixes: %s ." %(len(ases_sometimes_prepend_not_prepend), (ases_sometimes_prepend_not_prepend)))

new_origin_ases_prepending = list(set(confirmed_customers2_unique_origin_ases)-set(confirmed_customers1_unique_origin_ases))
print("%s number of unique customer ASNs of AS19551 that prepend their origin." %len(new_origin_ases_prepending))

prefixes_sometimes_prepend_not_prepend = set(confirmed_customers1_unique_prefixes) & set(confirmed_customers2_unique_prefixes)
print("%s number of prefixes are sometimes prepended by origin and sometimes not %s ." %(len(prefixes_sometimes_prepend_not_prepend), (prefixes_sometimes_prepend_not_prepend)))

new_prefixes_prepending = list(set(confirmed_customers2_unique_prefixes)-set(confirmed_customers1_unique_prefixes))
print("%s number of unique prefixes that contain AS19551 as provider after checking path prepending." %len(new_prefixes_prepending))

# # Find prefixes sizes, their numbers and IP versions
result = count_prefix_lengths_and_versions(new_prefixes_prepending)

# Print results
for version, lengths in result.items():
    print(f"{version}:")
    for length, count in lengths.items():
        print(f"  /{length}: {count} prefixes")
   


# After checking condition that AS19551 comes immediately after the siblings
df = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_not_as19551_01_jan_2017_v3.csv")
confirmed_customers3 = df.loc[(df['siblings'] == 1) & (df['new_provider_sibling_check'] == 19551)]

confirmed_customers3_unique_origin_ases = confirmed_customers3['origin_as'].unique()
confirmed_customers3_unique_prefixes = confirmed_customers3['prefix'].unique()

new_origin_ases_with_siblings = list(set(confirmed_customers3_unique_origin_ases)-set(confirmed_customers2_unique_origin_ases)-set(confirmed_customers1_unique_origin_ases))
print("%s number of unique customer ASNs of AS19551 that have siblings on their origin : %s." %(len(new_origin_ases_with_siblings), new_origin_ases_with_siblings))

new_prefixes_siblings = list(set(confirmed_customers3_unique_prefixes)- set(confirmed_customers2_unique_prefixes) - set(confirmed_customers1_unique_prefixes))
print("%s number of unique prefixes that contain AS19551 as provider after checking siblings." %len(new_prefixes_siblings))

# # Find prefixes sizes, their numbers and IP versions
result = count_prefix_lengths_and_versions(new_prefixes_siblings)

# Print results
for version, lengths in result.items():
    print(f"{version}:")
    for length, count in lengths.items():
        print(f"  /{length}: {count} prefixes")
   

5 number of ASes sometimes prepend their AS numbers and sometimes not for their different prefixes: {32748, 22612, 263477, 45753, 134974} .
7 number of unique customer ASNs of AS19551 that prepend their origin.
46 number of prefixes are sometimes prepended by origin and sometimes not {'202.61.65.0/24', '198.23.49.0/24', '103.70.77.0/24', '50.31.98.0/23', '162.210.97.0/24', '103.71.253.0/24', '162.213.248.0/24', '50.31.67.0/24', '208.100.53.0/24', '191.242.101.0/24', '67.202.92.0/24', '69.162.134.0/24', '198.23.61.0/24', '198.23.53.0/24', '208.117.45.0/24', '191.242.103.0/24', '208.117.38.0/24', '69.162.154.0/24', '198.23.62.0/24', '208.117.46.0/24', '198.23.51.0/24', '198.23.57.0/24', '198.23.48.0/24', '198.23.60.0/24', '198.23.63.0/24', '162.210.101.0/24', '162.210.103.0/24', '198.23.52.0/24', '50.31.100.0/24', '162.210.102.0/24', '198.23.54.0/24', '67.202.70.0/24', '198.23.50.0/24', '50.31.114.0/24', '50.31.74.0/24', '162.210.96.0/24', '162.210.100.0/24', '162.210.98.0/24', '198.23.5

In [ ]:
# Required to validate our finding with AS rank api
# Some bugs here
# new_origin_ases_without_siblings_not_immediate_provider = list(set(confirmed_customers3_unique_origin_ases) | set(confirmed_customers2_unique_origin_ases)| set(confirmed_customers1_unique_origin_ases) | set(customer_asns_not_sibling_not_immediate_provider))
# print("%s number of unique customer ASNs of AS32787 that contains that AS not as the second last hop on their origin : %s." %(len(new_origin_ases_without_siblings_not_immediate_provider), new_origin_ases_without_siblings_not_immediate_provider))




In [ ]:
# TO COMPLETE: Find the final list of customer ASNs and total prefixes
import pandas as pd
# df = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_not_as32787_01_jan_2024_v3.csv")
# df = df.loc[df["origin_as"] == 27533]
# unique_prefix = df["prefix"].unique()
# unique_prefix

# Find the number of prefixes originated by that ASN on that day looking into RIB file.
# The objective is to find what portion of prefixes are protected for each customer. 

# Characetrize the customers: For example: Bank customers protect all of their prefixes

# Do 

In [ ]:
# Program to find siblings of an ASN based on CAIDA AS2Org data
# Read line from 95721 from the file /h # format:aut|changed|aut_name|org_id|opaque_id|source
import gzip
import json

# Function to find siblings of a given ASN in a .jsonl.gz file, starting from a specific line
def find_siblings(asn_to_find):
    # Create a dictionary to group ASNs by organizationId
    org_id_map = {}
    
    file_path = '/home/shyam/jupy/ddos_scrubber/data/20241001.as-org2info.jsonl.gz'

    start_line = 95721
        
    # Open and read the .jsonl.gz file
    with gzip.open(file_path, 'rt') as f:  # 'rt' is for reading text mode
        for current_line, line in enumerate(f):
            if current_line < start_line:
                continue  # Skip lines until reaching the desired start_line
            
            record = json.loads(line)  # Parse each line as JSON
            
            asn = record['asn']
            organization_id = record['organizationId']
            
            if organization_id not in org_id_map:
                org_id_map[organization_id] = []
            org_id_map[organization_id].append(asn)
    
    # Find the organizationId of the given ASN
    org_id_of_asn = None
    for org_id, asns in org_id_map.items():
        if asn_to_find in asns:
            org_id_of_asn = org_id
            break

    # If ASN is not found, return an empty list
    if org_id_of_asn is None:
        return []

    # Return all ASNs that share the same organizationId, excluding the given ASN
    siblings = [asn for asn in org_id_map[org_id_of_asn] if asn != asn_to_find]
    return siblings

# Test the function with a .jsonl.gz file, starting from line 10
# asn_to_find = "23752"
# siblings = find_siblings(asn_to_find)
# print(f"Siblings of ASN {asn_to_find}: {siblings}")
# Siblings of ASN 23752: ['55726', '58413']

In [ ]:
# Find number of IRR invalid prefixes looking into RADB
import subprocess

# ASN to exclude
excluded_asn = "AS32787"

# Files to store results
no_asn_32787_file = "no_asn_32787.txt" # Prefixes without ASN 32787
multiple_asns_file = "multiple_asns.txt" # Prefixes containing multiple other ASNs including ASN 32787

# Open output files
with open(no_asn_32787_file, "w") as no_asn_file, open(multiple_asns_file, "w") as multi_asn_file:
    for prefix in prefixes:
        # Run whois query and capture output
        result = subprocess.run(["whois", "-h", "whois.radb.net", prefix], capture_output=True, text=True)
        
        # Extract origin ASNs from whois output
        origin_asns = set()
        for line in result.stdout.splitlines():
            if line.lower().startswith("origin"):
                asn = line.split()[-1]
                origin_asns.add(asn)

        # Check conditions and write to respective files
        if excluded_asn not in origin_asns:
            no_asn_file.write(f"{prefix} - Origin ASNs: {', '.join(origin_asns)}\n")
        
        if len(origin_asns) > 1:
            multi_asn_file.write(f"{prefix} - Multiple Origin ASNs: {', '.join(origin_asns)}\n")

print(f"Prefixes without ASN {excluded_asn} have been saved to {no_asn_32787_file}")
print(f"Prefixes with multiple origin ASNs have been saved to {multiple_asns_file}")

In [ ]:
# Program to find upstream provider using condition: 
# a. If the same origin AS is prepending, upstream AS is the next one after prepending. 
# b. If the origin AS has siblings, remove siblings. 
# CAVEAT: This program does not check as_path = [200, 300, 400, 400, 8074, 8075, 8075] where there is a repeatation of 
# new ASes other than siblings. E.g. AS400 is repeated here.

# Function to find the immediate provider ASN
def find_immediate_provider(as_path):
    if len(as_path) < 2:
        return None  # Not enough ASNs in path to determine provider

    last_origin_asn = as_path[-1]  # The last ASN is the origin ASN

    # Check for sequentially repeated ASNs
    repeated_asn = None
    for i in range(len(as_path) - 1, 0, -1):
        if as_path[i-1] == as_path[i]:
            repeated_asn = as_path[i]
        else:
            # If we find an ASN that is not the same as the repeated ASN,
            # and we have found a repeated ASN, return the one before the repeated ASN
            if repeated_asn is not None:
                return as_path[i-1]  # This is the upstream provider
            break

    # Do this check to call API only in case an origin has mutliple siblings

    # If no repeated ASN found or it's the only ASN in the path
    if repeated_asn is None:
        # Retrieve siblings for the last ASN (last origin in the AS path)
        try:
            sibling_list = find_siblings(last_origin_asn)
        except Exception as e:
            print(f"Error retrieving siblings: {e}")
            return None  # Handle API failure appropriately

        # Traverse the AS path 
        for i in range(len(as_path) - 1, 0, -1):
            if str(as_path[i-1]) not in sibling_list:
                return as_path[i-1]  # The first non-sibling ASN is the upstream provider

    return as_path[1]  # If all checks fail, return the second ASN by default


# Test cases based on your examples
# as_path = [400, 100, 300, 300]  # Case 1: No repetition, no siblings
# as_path = ["200", "300", "400", "400", "6584", "8074", "8075"]  # Case 2: Sequentially repeated, should return 300


# # # Finding immediate providers
# print("Immediate provider for AS path:", find_immediate_provider(as_path))  # Expected Output: 300


In [22]:
# Program to find siblings of an ASN based on CAIDA AS2Org data
# Read line from 95721 from the file /h # format:aut|changed|aut_name|org_id|opaque_id|source
import gzip
import json

# Function to find siblings of a given ASN in a .jsonl.gz file, starting from a specific line
def find_siblings(asn_to_find):
    # Create a dictionary to group ASNs by organizationId
    org_id_map = {}
    
    file_path = '/home/shyam/jupy/ddos_scrubber/data/20241001.as-org2info.jsonl.gz'

    start_line = 95721
        
    # Open and read the .jsonl.gz file
    with gzip.open(file_path, 'rt') as f:  # 'rt' is for reading text mode
        for current_line, line in enumerate(f):
            if current_line < start_line:
                continue  # Skip lines until reaching the desired start_line
            
            record = json.loads(line)  # Parse each line as JSON
            
            asn = record['asn']
            organization_id = record['organizationId']
            
            if organization_id not in org_id_map:
                org_id_map[organization_id] = []
            org_id_map[organization_id].append(asn)
    
    # Find the organizationId of the given ASN
    org_id_of_asn = None
    for org_id, asns in org_id_map.items():
        if asn_to_find in asns:
            org_id_of_asn = org_id
            break

    # If ASN is not found, return an empty list
    if org_id_of_asn is None:
        return []

    # Return all ASNs that share the same organizationId, excluding the given ASN
    siblings = [asn for asn in org_id_map[org_id_of_asn] if asn != asn_to_find]
    return siblings

# Test the function with a .jsonl.gz file, starting from line 10
asn_to_find = "23752"
siblings = find_siblings(asn_to_find)
print(f"Siblings of ASN {asn_to_find}: {siblings}")
# Siblings of ASN 23752: ['55726', '58413']

Siblings of ASN 23752: ['55726', '58413']
